In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML


# --------------------------------
# FIGURE SETUP
# --------------------------------

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.set_xlim(0,500)
ax.set_ylim(-50,100)
ax.set_zlim(0,20)

ax.set_xlabel("X (miles)")
ax.set_ylabel("Y (miles)")
ax.set_zlabel("Z (miles)")

ax.set_box_aspect([1,1,0.2])

fig.patch.set_facecolor("lightblue") #whole canvas
ax.set_facecolor("cyan") #background of 3D boxes
ax.xaxis.pane.set_facecolor("lightgreen") #background of x-axis plane
ax.yaxis.pane.set_facecolor("lightgreen") #background of y-axis plane
ax.zaxis.pane.set_facecolor("blue") #background of z-axis plane


# --------------------------------
# HELICOPTER PATH FUNCTIONS
# --------------------------------

def H1_path(t):

    x = 6 + 40*t
    y = -3 + 10*t
    z = -3 + 2*t

    return np.array([x,y,z])


def H2_path(t):

    x = 6 + 110*t
    y = -3 + 4*t
    z = -3 + t

    return np.array([x,y,z])


# --------------------------------
# IMPORTANT TIMES
# --------------------------------

t_stop = 4
t_notice = 6


H2_stop = np.array([446,13,1])
H2_land = np.array([446,13,0])

H1_notice = H1_path(t_notice)


# --------------------------------
# OBJECTS
# --------------------------------

H1_point, = ax.plot([],[],[],'bo',label="H1")
H2_point, = ax.plot([],[],[],'ro',label="H2")

H1_traj, = ax.plot([],[],[],'b--',alpha=0.5)
H2_traj, = ax.plot([],[],[],'r--',alpha=0.5)

rescue_line, = ax.plot([],[],[],'g',linewidth=2,label="Rescue path")

ax.legend()


# --------------------------------
# STORAGE FOR TRAJECTORIES
# --------------------------------

H1_x,H1_y,H1_z = [],[],[]
H2_x,H2_y,H2_z = [],[],[]


# --------------------------------
# UPDATE FUNCTION
# --------------------------------

def update(frame):

    t = frame*0.1

    ax.set_title(f"Helicopter Rescue Animation\n time = {t:.2f} hours")

    # ------------------------------
    # PHASE 1 : both flying
    # ------------------------------

    if t <= t_stop:

        H1 = H1_path(t)
        H2 = H2_path(t)

    # ------------------------------
    # PHASE 2 : H2 stops
    # ------------------------------

    elif t_stop < t <= t_notice:

        H1 = H1_path(t)
        H2 = H2_stop

    # ------------------------------
    # PHASE 3 : rescue flight
    # ------------------------------

    else:

        H2 = H2_land

        direction = H2 - H1_notice

        s = min((t - t_notice)/1.37 ,1)

        H1 = H1_notice + s*direction


    # store trajectories

    H1_x.append(H1[0])
    H1_y.append(H1[1])
    H1_z.append(H1[2])

    H2_x.append(H2[0])
    H2_y.append(H2[1])
    H2_z.append(H2[2])


    # update positions

    H1_point.set_data([H1[0]],[H1[1]])
    H1_point.set_3d_properties([H1[2]])

    H2_point.set_data([H2[0]],[H2[1]])
    H2_point.set_3d_properties([H2[2]])


    # update paths

    H1_traj.set_data(H1_x,H1_y)
    H1_traj.set_3d_properties(H1_z)

    H2_traj.set_data(H2_x,H2_y)
    H2_traj.set_3d_properties(H2_z)


    # draw rescue line

    if t > t_notice:

        rescue_line.set_data([H1[0],H2[0]],
                     [H1[1],H2[1]])

        rescue_line.set_3d_properties([H1[2],H2[2]])    

    return H1_point,H2_point,H1_traj,H2_traj,rescue_line


# --------------------------------
# ANIMATION
# --------------------------------

ani = FuncAnimation(
    fig,
    update,
    frames=120,
    interval=80
)

plt.close(fig)

HTML(ani.to_jshtml())

ani.save(
    "/home/shubh/projects/3d-anim/assets/gifs/helicopter_rescue.gif",
    writer="pillow",
    fps=24,
    dpi=200,
    bitrate=2400
)